In [ ]:
# ============================================================
# CELL 0: Install Requirements (Run this first on Colab)
# ============================================================
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

packages = [
    "transformers==4.40.0",
    "peft",           # LoRA support
    "datasets",
    "seqeval",
    "tqdm",
    "rich",
    "pypdf2",
    "pdfplumber",
    "scikit-learn",
    "matplotlib",
    "seaborn",
    "tensorflow>=2.13",
    "tf-keras",
]

for pkg in packages:
    try:
        install(pkg)
    except Exception as e:
        print(f"Warning: could not install {pkg}: {e}")

print("✅ All requirements installed!")


✅ All requirements installed!


<a id="1"></a>
<h1 style="background: #FFC07F; border: 0; color: #2F2E41;
    box-shadow: 4px 4px 8px rgba(0, 0, 0, 0.3);
    padding: 10px; border-radius: 10px; margin: 15px 0;">
    <center style="color: #2F2E41;">1. Overview</center>
</h1>

<a id="1-1"></a>

## 1.1. Transformer Network Application: Named-Entity Recognition (NER)

In this study, we delve into a practical application of the transformer architecture, focusing on Named-Entity Recognition (NER). This exploration builds upon the transformer model and applies it to a real-world task.

NER is a foundational task in natural language processing (NLP) that involves identifying and classifying named entities such as names, dates, locations, and organizations within text. By leveraging pre-trained transformer models, we can achieve state-of-the-art performance in this task with minimal training data, thanks to transfer learning.

In the subsequent sections, we will demonstrate how to preprocess textual data, tokenize it for transformer models, and fine-tune a pre-trained model specifically for the NER task. This approach highlights the versatility and power of transformer networks in handling diverse NLP challenges.

<a id="1-2"></a>

## 1.2. Objectives

Through this study, we aim to:

- Utilize tokenizers and pre-trained models provided by the HuggingFace library.
- Fine-tune a pre-trained transformer model to perform Named-Entity Recognition effectively.


In [ ]:
!pip uninstall -y transformers peft accelerate
!pip install transformers==4.40.2 peft==0.10.0 accelerate==0.29.3

Found existing installation: transformers 4.40.0
Uninstalling transformers-4.40.0:
  Successfully uninstalled transformers-4.40.0
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.6/297.6 kB 11.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.


<a id="2"></a>
<h1 style="background: #FFC07F; border: 0; color: #2F2E41;
    box-shadow: 4px 4px 8px rgba(0, 0, 0, 0.3);
    padding: 10px; border-radius: 10px; margin: 15px 0;">
    <center style="color: #2F2E41;">2. Imports</center>
</h1>

In [ ]:
# ============================================================
# CELL 1: Imports
# ============================================================
import pandas as pd
import numpy as np
import tensorflow as tf
import json
import random
import logging
import re
import os
import io

from tqdm.rich import tqdm
from collections import Counter

from tensorflow.keras.preprocessing.sequence import pad_sequences

# DistilBERT (original model - kept as-is)
from transformers import DistilBertTokenizerFast
from transformers import TFDistilBertForTokenClassification

# BERT model (new addition)
from transformers import BertTokenizerFast
from transformers import TFBertForTokenClassification

# LoRA (new addition)
from peft import get_peft_model, LoraConfig, TaskType
from transformers import AutoModelForTokenClassification, AutoTokenizer

# Evaluation
from sklearn.metrics import classification_report

from matplotlib import pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

tf.get_logger().setLevel('ERROR')
print("✅ Imports successful!")


✅ Imports successful!


<a id="3"></a>
<h1 style="background: #FFC07F; border: 0; color: #2F2E41;
    box-shadow: 4px 4px 8px rgba(0, 0, 0, 0.3);
    padding: 10px; border-radius: 10px; margin: 15px 0;">
    <center style="color: #2F2E41;">3. Data Preparation</center>
</h1>

<a id='3-1'></a>

## 3.1. Named-Entity Recognition to Process Resumes

When working with large volumes of unstructured text, Named-Entity Recognition (NER) plays a crucial role in identifying and classifying key information. For example, in the sentence _"Jane visits Africa in September"_, NER can detect _"Jane"_, _"Africa"_, and _"September"_ as entities and classify them as `Person`, `Location`, and `Time`, respectively.

In this application, we use a variation of the transformer model to analyze a dataset of resumes. Our goal is to identify and classify relevant entities such as the names of companies, skills, degrees, and other pertinent details.

<a id='3-2'></a>

## 3.2. Dataset Cleaning and Preprocessing

Before training the transformer model, it is essential to preprocess the dataset. The dataset contains resumes structured as JSON objects with annotations marking the entities. We clean the data and organize it into a format suitable for training an NER model.

<a id='3-2-1'></a>

### 3.2.1. Data Loading and Initial Exploration

In [ ]:
# ============================================================
# DATA LOADING - Supports JSON (lines), CSV formats
# ============================================================

def load_ner_dataset(filepath):
    """
    Smart loader that handles multiple CSV/JSON formats automatically.
    Supports:
      - JSON lines (one JSON object per line) with 'content' + 'annotation' columns
      - CSV with 'content' + 'annotation' columns
      - CSV with pre-tokenized 'sentence_id', 'word', 'tag' columns (CoNLL style)
    """
    ext = os.path.splitext(filepath)[-1].lower()

    # --- JSON lines format (original Kaggle format) ---
    if ext == '.json':
        df = pd.read_json(filepath, lines=True)
        if 'extras' in df.columns:
            df = df.drop(['extras'], axis=1)
        df['content'] = df['content'].str.replace("\n", " ")
        print(f"✅ Loaded JSON: {len(df)} rows | columns: {list(df.columns)}")
        return df, 'json_annotation'

    # --- CSV formats ---
    elif ext == '.csv':
        df = pd.read_csv(filepath)
        df.columns = df.columns.str.strip().str.lower()

        # Format 1: content + annotation columns (same as JSON)
        if 'content' in df.columns and 'annotation' in df.columns:
            df['content'] = df['content'].astype(str).str.replace("\n", " ")
            # annotation might be stored as string - try to parse
            import ast
            df['annotation'] = df['annotation'].apply(
                lambda x: ast.literal_eval(x) if isinstance(x, str) else x
            )
            print(f"✅ Loaded CSV (annotation format): {len(df)} rows")
            return df, 'json_annotation'

        # Format 2: CoNLL-style (sentence_id, word, tag)
        elif all(c in df.columns for c in ['word', 'tag']):
            sid_col = next((c for c in ['sentence_id','sentence','sent_id'] if c in df.columns), None)
            if sid_col is None:
                # Create sentence_id from blank lines or sequential
                df['sentence_id'] = (df['word'].isna() | (df['word'] == '')).cumsum()
            else:
                df['sentence_id'] = df[sid_col]
            df = df.dropna(subset=['word','tag'])
            print(f"✅ Loaded CSV (CoNLL format): {len(df)} rows, "
                  f"{df['sentence_id'].nunique()} sentences, "
                  f"tags: {df['tag'].unique().tolist()}")
            return df, 'conll'

        # Format 3: Resume text in one column
        elif 'resume' in df.columns or 'text' in df.columns:
            text_col = 'resume' if 'resume' in df.columns else 'text'
            df = df.rename(columns={text_col: 'content'})
            df['content'] = df['content'].astype(str).str.replace("\n", " ")
            df['annotation'] = [[] for _ in range(len(df))]  # no annotations
            print(f"✅ Loaded CSV (text-only): {len(df)} rows")
            return df, 'json_annotation'

        else:
            raise ValueError(f"Unrecognized CSV format. Columns: {list(df.columns)}")
    else:
        raise ValueError(f"Unsupported file format: {ext}. Use .json or .csv")


# ---- Load the dataset (change path as needed) ----
DATA_PATH = "/content/Entity Recognition in Resumes.json"

df_data, data_format = load_ner_dataset(DATA_PATH)
df_data.head()


✅ Loaded JSON: 220 rows | columns: ['content', 'annotation']


,content,annotation
0,Abhishek Jha Application Development Associate...,"[{'label': ['Skills'], 'points': [{'start': 12..."
1,Afreen Jamadar Active member of IIIT Committee...,"[{'label': ['Email Address'], 'points': [{'sta..."
2,"Akhil Yadav Polemaina Hyderabad, Telangana - E...","[{'label': ['Skills'], 'points': [{'start': 37..."
3,Alok Khandai Operational Analyst (SQL DBA) Eng...,"[{'label': ['Skills'], 'points': [{'start': 80..."
4,Ananya Chavan lecturer - oracle tutorials Mum...,"[{'label': ['Degree'], 'points': [{'start': 20..."


**Explanation**:  
1. The dataset is read from a JSON file (`ner.json`) and loaded into a pandas DataFrame `df_data`. `df_data` holds various columns. The key columns in this case include:
     - **`content`**: The actual textual data (such as resumes).
     - **`annotation`**: A column containing the labeled named-entities, which are used to classify parts of the text (like names, dates, skills, etc.).
2. The `extras` column, containing non-essential information, is dropped to reduce noise.
3. Newline characters in the `content` column are replaced with spaces to ensure consistent text formatting for further processing.

#### **Explanation of `annotation` Structure**

1. **List of Dictionaries**:
   - The `annotation` column contains a **list** of dictionaries, where each dictionary represents a single named-entity annotation. Each annotation contains:
     - **`label`**: A list of labels representing the type of entity (e.g., `Skills`, `College Name`, etc.).
     - **`points`**: A list of dictionaries that contain the start and end  character indices of the entity within the text, as well as the entity text itself.

2. **Example Annotations**:
   - **First Annotation** (Skills):
     - **`label`:** `["Skills"]` indicates that the entity is a skill.
     - **`points`:** Contains a dictionary with:
       - **`start`:** The starting index (1295) of the entity in the text.
       - **`end`:** The ending index (1621) of the entity in the text.
       - **`text`:** The actual entity text, which in this case is a list of skills like "Programming language: C, C++, Java" and other technical and non-technical skills.
   
   - **Second Annotation** (Skills):
     - **`label`:** `["Skills"]` indicates that this entity also refers to a skill.
     - **`points`:** Contains:
       - **`start`:** 993
       - **`end`:** 1153
       - **`text`:** A list of skills along with experience durations (e.g., "C (Less than 1 year), Database (Less than 1 year), etc.").
       
3. **Purpose**:
   - The annotations in the `annotation` column are used for Named Entity Recognition (NER), where entities such as `Skills`, `College Name`, `Companies worked at`, and other aspects of a resume are automatically labeled and identified.
   
   - These annotations are crucial for training and fine-tuning NER models, which are designed to automatically identify these entities in unseen resume data.

The following code retrieves the **annotation** for the first entry (or row) in the DataFrame `df_data` specifically from the **`annotation`** column.

In [ ]:
df_data.iloc[0]['annotation']

[{'label': ['Skills'],
  'points': [{'start': 1295,
    'end': 1621,
    'text': '\n• Programming language: C, C++, Java\n• Oracle PeopleSoft\n• Internet Of Things\n• Machine Learning\n• Database Management System\n• Computer Networks\n• Operating System worked on: Linux, Windows, Mac\n\nNon - Technical Skills\n\n• Honest and Hard-Working\n• Tolerant and Flexible to Different Situations\n• Polite and Calm\n• Team-Player'}]},
 {'label': ['Skills'],
  'points': [{'start': 993,
    'end': 1153,
    'text': 'C (Less than 1 year), Database (Less than 1 year), Database Management (Less than 1 year),\nDatabase Management System (Less than 1 year), Java (Less than 1 year)'}]},
 {'label': ['College Name'],
  'points': [{'start': 939, 'end': 956, 'text': 'Kendriya Vidyalaya'}]},
 {'label': ['College Name'],
  'points': [{'start': 883, 'end': 904, 'text': 'Woodbine modern school'}]},
 {'label': ['Graduation Year'],
  'points': [{'start': 856, 'end': 860, 'text': '2017\n'}]},
 {'label': ['College 

<a id='3-2-2'></a>

### 3.2.2. Merging Overlapping Entity Intervals

In [ ]:
def merge_intervals(intervals): # intervals = [(start, end, label)]
    """Merge overlapping entity intervals, preserving entity labels."""
    # Sort intervals by the start index to facilitate merging
    sorted_intervals = sorted(intervals, key=lambda tup: tup[0])
    merged = []

    for current in sorted_intervals:
        if not merged:
            merged.append(current)
        else:
            previous = merged[-1]
            if current[0] <= previous[1]:
                if previous[2] == current[2]:
                    # Merge intervals if they overlap and have the same label
                    merged[-1] = (previous[0], max(previous[1], current[1]), previous[2])
                else:
                    # Merge intervals if they overlap but have different labels
                    merged[-1] = (previous[0], max(previous[1], current[1]), current[2])
            else:
                # No overlap, just append the current interval
                merged.append(current)
    return merged

**Explanation**:  
The function `merge_intervals` is designed to **merge overlapping or adjacent intervals** (in this case, entity spans in text) while maintaining their associated **labels** (e.g., "Person," "Location," etc.). This is important in Named-Entity Recognition (NER), where entities can sometimes span across adjacent or overlapping segments, and we need to group them correctly.

1. **Input Structure:**
   - The function takes a list of intervals in the form of **tuples**: `(start, end, label)`.
     - `start`: The starting index of the entity in the text.
     - `end`: The ending index of the entity in the text (non-inclusive).
     - `label`: The type of entity, such as "Person", "Location", or "Organization".

   Example input:

   ```python
   intervals = [(5, 10, 'Person'), (8, 12, 'Person'), (13, 15, 'Location')]
   ```

   Here, we have three intervals:
   - The first interval spans from index 5 to 10 with the label "Person."
   - The second interval spans from index 8 to 12 with the label "Person."
   - The third interval spans from index 13 to 15 with the label "Location."

2. **Sorting Intervals:**
   - The intervals are first **sorted by the start index** using the `sorted()` function:

   ```python
   sorted_intervals = sorted(intervals, key=lambda tup: tup[0])
   ```

   Sorting ensures that the intervals are in order from the start of the text to the end, which is crucial for efficient merging.

   After sorting, the intervals look like this:

   ```python
   sorted_intervals = [(5, 10, 'Person'), (8, 12, 'Person'), (13, 15, 'Location')]
   ```

3. **Merging Intervals:**
   - The main purpose of this function is to **merge overlapping or adjacent intervals**. The algorithm iterates over the sorted intervals and compares each interval (`current`) with the **last merged interval** (`previous`).

   **If the `current` interval overlaps with the `previous` interval:**
   - **Case 1: Same Label** – If the labels of the two intervals are the same (i.e., they refer to the same type of entity), we merge them into a single interval that spans from the start of the `previous` interval to the end of the `current` interval.
     - This is done by updating the last merged interval (`merged[-1]`) to have the start of the `previous` interval and the maximum of the `end` values of both intervals.
   
   **Example:**
   
   ```python
   # Merging (5, 10, 'Person') and (8, 12, 'Person')
   merged[-1] = (5, max(10, 12), 'Person')  # (5, 12, 'Person')
   ```

   - **Case 2: Different Labels** – If the labels are different (i.e., they refer to different types of entities), we merge the intervals but take the label of the `current` interval for the merged entity. This is because, in some contexts, it may make sense to prioritize the entity type of the `current` interval over the `previous` one.

   **Example:**
   
   ```python
   # Merging (10, 15, 'Person') and (13, 15, 'Location')
   merged[-1] = (10, max(15, 15), 'Location')  # (10, 15, 'Location')
   ```

   **If the `current` interval does not overlap with the `previous` interval:**
   - The interval is simply appended to the `merged` list as it doesn't need to be merged.

4. **Returning the Merged Intervals:**
   - Once all intervals are processed, the function returns the list of merged intervals.

##### **Key Concepts**

- **Overlapping intervals:** When two intervals overlap (i.e., the `start` of the current interval is less than or equal to the `end` of the previous interval), they can be merged.
- **Same Label vs. Different Label:** When merging overlapping intervals, if the labels are the same, the intervals are merged into one. If the labels are different, the `current` label takes priority.
- **Efficiency:** Sorting the intervals beforehand ensures that merging is done in a single pass through the list, which is efficient.


<a id='3-2-3'></a>

### 3.2.3. Extracting Entity Annotations

In [ ]:
def get_entities(df):
    """Extract and merge entity annotations for each resume."""
    entities = []

    for i in range(len(df)):
        entity_list = []
        for annot in df['annotation'][i]:
            try:
                label = annot['label'][0]
                start = annot['points'][0]['start']
                end = annot['points'][0]['end'] + 1 # Adjusting the end index by +1 to include the last character
                # as python slicing is non-inclusive
                entity_list.append((start, end, label))
            except:
                pass
        merged_entities = merge_intervals(entity_list)
        entities.append(merged_entities)

    return entities

**Explanation**:  
1. The function iterates through each row of the DataFrame to extract annotations.
2. Start and end positions, along with the entity label, are parsed and stored.
 - The end index is adjusted  by +1 to include the last character as python slicing is non-inclusive.
3. Overlapping intervals are resolved using the `merge_intervals` function.

In [ ]:
df_data['entities'] = get_entities(df_data)
df_data.head()

,content,annotation,entities
0,Abhishek Jha Application Development Associate...,"[{'label': ['Skills'], 'points': [{'start': 12...","[(0, 12, Name), (13, 46, Designation), (49, 58..."
1,Afreen Jamadar Active member of IIIT Committee...,"[{'label': ['Email Address'], 'points': [{'sta...","[(0, 14, Name), (62, 68, Location), (104, 148,..."
2,"Akhil Yadav Polemaina Hyderabad, Telangana - E...","[{'label': ['Skills'], 'points': [{'start': 37...","[(0, 21, Name), (22, 31, Location), (65, 117, ..."
3,Alok Khandai Operational Analyst (SQL DBA) Eng...,"[{'label': ['Skills'], 'points': [{'start': 80...","[(0, 12, Name), (13, 51, Designation), (54, 60..."
4,Ananya Chavan lecturer - oracle tutorials Mum...,"[{'label': ['Degree'], 'points': [{'start': 20...","[(0, 13, Name), (14, 22, Designation), (24, 41..."


<a id='3-2-4'></a>

### 3.2.4. Converting to spaCy Format

Now we define a function `convert_dataturks_to_spacy` which converts annotations from the Dataturks JSON format into a format compatible with spaCy for training. Each entry consists of the raw text and its corresponding entity annotations. function `convert_dataturks_to_spacy` follows the approach:

1. **Initialize Data Storage**: Create containers for storing processed data and raw lines from the file.
2. **Read and Parse File**: Open the file and read each line as JSON data.
3. **Extract Text Content**: Clean and prepare the `content` field from the JSON object.
4. **Check and Process Annotations**: If annotations exist:
    - Iterate over each annotation.
    - Extract and clean the labeled spans.
    - Handle any discrepancies in indices due to whitespace.
5. **Store Entities**: Format and store the labeled data as `(start, end, label)` tuples.
6. **Combine Text and Entities**: Append the processed text and entities as a tuple into `training_data`.
7. **Handle Exceptions**: Log any errors and return `None` if processing fails.

In [ ]:
def convert_dataturks_to_spacy(dataturks_JSON_FilePath):
    try:
        # Step 1: Initialize data storage
        training_data = []
        lines = []

        # Step 2: Read and parse the file
        with open(dataturks_JSON_FilePath, 'r') as f:
            lines = f.readlines()

        # Step 3: Process each line of the file
        for line in lines:
            # Parse JSON data for each line
            data = json.loads(line)

            # Extract and clean the text content
            text = data['content'].replace("\n", " ")
            entities = []  # Initialize entities for this text

            # Step 4: Check for annotations and process if they exist
            data_annotations = data.get('annotation')
            if data_annotations is not None:
                for annotation in data_annotations:
                    # Extract the first annotation point (start, end, and text)
                    point = annotation['points'][0]
                    labels = annotation['label']

                    # Handle both single and multiple labels
                    if not isinstance(labels, list):
                        labels = [labels]

                    for label in labels:
                        # Extract start, end indices, and the annotated text
                        point_start = point['start']
                        point_end = point['end']
                        point_text = point['text']

                        # Step 5: Adjust indices to account for whitespace
                        lstrip_diff = len(point_text) - len(point_text.lstrip())
                        rstrip_diff = len(point_text) - len(point_text.rstrip())
                        if lstrip_diff != 0:
                            point_start = point_start + lstrip_diff
                        if rstrip_diff != 0:
                            point_end = point_end - rstrip_diff

                        # Append the entity tuple (start, end, label)
                        entities.append((point_start, point_end + 1, label))

            # Step 6: Store the processed text and entities
            training_data.append((text, {"entities": entities}))

        # Step 7: Return the formatted training data
        return training_data

    except Exception as e:
        # Step 8: Handle exceptions
        logging.exception(
            "Unable to process " + dataturks_JSON_FilePath + "\n" + "error = " + str(e)
        )
        return None

<a id='3-2-5'></a>

### 3.2.5. Cleaning Entity Spans

Now we define a function `trim_entity_spans` which removes leading and trailing whitespace from entity spans in a dataset formatted for spaCy. This cleaning ensures that the entity annotations accurately reflect the positions of meaningful characters in the text, avoiding mismatches due to extra spaces. The function follows the approach:

1. **Compile Invalid Tokens**: Create a regular expression to match whitespace characters (`\s`), which will help identify unwanted spaces in the text.
2. **Initialize Cleaned Data Storage**: Create a container to store the cleaned text and adjusted entity annotations.
3. **Iterate Over the Dataset**: For each text and its corresponding annotations:
   - Extract the list of entity spans.
   - Initialize a container for validated entities.
4. **Validate Entity Spans**: For each entity:
   - Adjust the start index to skip leading whitespace until a non-whitespace character is found.
   - Adjust the end index to skip trailing whitespace by moving backward to the nearest non-whitespace character.
   - Append the adjusted span and its label to the validated entities list.
5. **Store Cleaned Data**: Combine the cleaned text and validated entities into the cleaned dataset container.
6. **Return Cleaned Dataset**: Output the fully processed dataset with whitespace-trimmed entity spans.

In [ ]:
def trim_entity_spans(data: list) -> list:
    """Removes leading and trailing white spaces from entity spans.

    Args:
        data (list): The data to be cleaned in spaCy JSON format.

    Returns:
        list: The cleaned data.
    """
    # Step 1: Compile a regular expression to identify whitespace tokens
    invalid_span_tokens = re.compile(r'\s')

    # Step 2: Initialize storage for cleaned data
    cleaned_data = []

    # Step 3: Iterate over the dataset
    for text, annotations in data:
        entities = annotations['entities']  # Extract the entity list
        valid_entities = []  # Initialize storage for validated entities

        # Step 4: Validate and adjust each entity span
        for start, end, label in entities:
            valid_start = start  # Adjusted start index
            valid_end = end      # Adjusted end index

            # Skip leading whitespace
            while valid_start < len(text) and invalid_span_tokens.match(
                    text[valid_start]):
                valid_start += 1

            # Skip trailing whitespace
            while valid_end > 1 and invalid_span_tokens.match(
                    text[valid_end - 1]):
                valid_end -= 1

            # Append the cleaned span and its label
            valid_entities.append([valid_start, valid_end, label])

        # Step 5: Store the cleaned text and adjusted entities
        cleaned_data.append([text, {'entities': valid_entities}])

    # Step 6: Return the cleaned dataset
    return cleaned_data

Now we apply the functions `convert_dataturks_to_spacy` and `trim_entity_spans` sequentially to process a dataset from a file named `"ner.json"`. The combined workflow ensures the data is converted into a spaCy-compatible format and cleaned for any discrepancies in entity span annotations. The steps are as follows:

1. **Convert Dataturks Data to spaCy Format**:
   - Use the `convert_dataturks_to_spacy` function to parse the Dataturks JSON file.
   - Extract raw text and its associated entity annotations.
   - Format the data into a structure compatible with spaCy's training requirements.

2. **Clean Entity Spans**:
   - Pass the spaCy-formatted data to the `trim_entity_spans` function.
   - Remove any leading or trailing whitespace from the entity spans.
   - Adjust the start and end indices to match the non-whitespace content accurately.

3. **Output Cleaned Data**:
   - The resulting dataset will contain texts and their precise entity annotations, ready for spaCy model training.

In [ ]:
data = trim_entity_spans(convert_dataturks_to_spacy(DATA_PATH))


In [ ]:
data[0]

["Abhishek Jha Application Development Associate - Accenture  Bengaluru, Karnataka - Email me on Indeed: indeed.com/r/Abhishek-Jha/10e7a8cb732bc43a  • To work for an organization which provides me the opportunity to improve my skills and knowledge for my individual and company's growth in best possible ways.  Willing to relocate to: Bangalore, Karnataka  WORK EXPERIENCE  Application Development Associate  Accenture -  November 2017 to Present  Role: Currently working on Chat-bot. Developing Backend Oracle PeopleSoft Queries for the Bot which will be triggered based on given input. Also, Training the bot for different possible utterances (Both positive and negative), which will be given as input by the user.  EDUCATION  B.E in Information science and engineering  B.v.b college of engineering and technology -  Hubli, Karnataka  August 2013 to June 2017  12th in Mathematics  Woodbine modern school  April 2011 to March 2013  10th  Kendriya Vidyalaya  April 2001 to March 2011  SKILLS  C (Le

<a id='3-2-6'></a>

### 3.2.6. Further Cleaning the Dataset

1. **Initialize Storage**:
   - Create an empty DataFrame `cleanedDF` to store lists of entity labels for each word in the resumes.
   - Initialize a variable `total_words` to track the total number of words processed.

2. **Iterate Over Resumes**:
   - For each resume in the input dataset, extract the resume text and the associated entity data.

3. **Initialize Word Tracking**:
   - Calculate the total number of words in the resume and the length of the resume text in characters.
   - Create a list `entity_labels` initialized to `"Empty"` for each word in the resume to store entity labels.

4. **Process Characters to Identify Words**:
   - Iterate through the characters in the resume text.
   - Identify word boundaries by detecting spaces. For each word:
     - Check if the word falls within any entity span.
     - If it does, assign the corresponding entity label to the word in `entity_labels`.

5. **Handle the Last Word**:
   - Ensure the last word in the resume (not followed by a space) is processed.
   - If it matches an entity span, assign the corresponding entity label.

6. **Store Results**:
   - Append the `entity_labels` list for the resume as a new row in the DataFrame `cleanedDF`.

7. **Return Results**:
   - Return the cleaned DataFrame containing entity labels for all resumes.


In [ ]:
def clean_dataset(data):
    """
    Cleans the input dataset by mapping each word in the resume text to its corresponding entity label or 'Empty'.

    Args:
        data (list): A list where each element contains a resume text and an entity dictionary.
                     Example format:
                     [
                        [resume_1, {'entities': [[start1, end1, 'Entity1'], ...]}],
                        [resume_2, {'entities': [[start2, end2, 'Entity2'], ...]}],
                     ]

    Returns:
        pd.DataFrame: A DataFrame with one column containing lists of entity labels for each word in the resume text.
    """
    # Step 1: Initialize an empty DataFrame for storing the cleaned entity labels
    cleanedDF = pd.DataFrame(columns=["entity_labels"])
    total_words = 0  # Tracks the total number of processed words across all resumes

    # Step 2: Process each resume in the input dataset
    for idx in tqdm(range(len(data)), desc="Processing resumes"):
        resume_text = data[idx][0]  # Extract the resume text
        entity_data = data[idx][1]['entities']  # Extract the entity data
        num_words = len(resume_text.split())  # Total number of words in the resume
        len_text = len(resume_text)  # Length of the resume text in characters

        # Step 3: Prepare a list initialized to "Empty" for each word in the resume
        entity_labels = ["Empty"] * num_words
        word_index = 0  # Tracks the position of the current word in the resume
        start_char = 0  # Tracks the starting character of the current word

        # Step 4: Iterate through the characters in the resume to identify word boundaries
        for char_idx in range(len_text):
            # Detect the end of a word when encountering a space
            if resume_text[char_idx] == " " and (char_idx + 1 < len_text and resume_text[char_idx + 1] != " "):
                # Check if the current word falls within any entity span
                for entity in reversed(entity_data):  # Process entities in reverse order for efficiency
                    entity_start, entity_end, entity_label = entity
                    entity_start = int(entity_start)
                    entity_end = int(entity_end)
                    if start_char >= entity_start and char_idx <= entity_end:
                        entity_labels[word_index] = entity_label  # Assign the entity label to the word
                        break

                # Move to the next word and update the starting character index
                word_index += 1
                start_char = char_idx + 1

        # Step 5: Handle the last word in the resume (not followed by a space)
        for entity in reversed(entity_data):
            entity_start, entity_end, entity_label = entity
            entity_start = int(entity_start)
            entity_end = int(entity_end)
            if start_char >= entity_start and len_text <= entity_end:
                entity_labels[word_index] = entity_label
                break

        # Step 6: Append the processed entity labels for this resume to the DataFrame
        cleanedDF = pd.concat(
            [cleanedDF, pd.DataFrame([[entity_labels]], columns=cleanedDF.columns)],
            ignore_index=True
        )
        total_words += num_words  # Increment the total word count

    # Step 7: Return the final cleaned DataFrame
    return cleanedDF

In [ ]:
cleanedDF = clean_dataset(data)

Output()

Lets take a look at our cleaned dataset and the categories the named-entities are matched to, or 'tags'.

In [ ]:
cleanedDF.head()

,entity_labels
0,"[Name, Name, Designation, Designation, Designa..."
1,"[Name, Name, Empty, Empty, Empty, Empty, Empty..."
2,"[Name, Name, Name, Empty, Empty, Empty, Empty,..."
3,"[Name, Name, Designation, Designation, Designa..."
4,"[Name, Name, Designation, Empty, Companies wor..."


<a id='3-2-7'></a>

### 3.2.7. Padding and Generating Tags

In this section, we focus on creating a comprehensive list of unique tags derived from the cleaned dataset, which represent the named entities to be matched and processed. These tags will be used for mapping between textual representations and numerical IDs, facilitating efficient handling in subsequent steps.

#### Objective

- Extract a set of unique tags from the cleaned dataset.
- Create mappings:
  - **`tag2id`**: Maps each unique tag to a unique numerical ID.
  - **`id2tag`**: Maps numerical IDs back to their corresponding tags.

#### Implementation

Below is the implementation for generating the unique tags and the corresponding mappings.

In [ ]:
# Extract unique tags from the cleaned dataset
unique_tags = set(cleanedDF['entity_labels'].explode().unique())

# Create a mapping from tags to unique numerical IDs
tag2id = {tag: id for id, tag in enumerate(unique_tags)}

# Create a reverse mapping from numerical IDs to tags
id2tag = {id: tag for tag, id in tag2id.items()}

#### Explanation of Code Logic

1. **Extract Unique Tags**:
   - `cleanedDF['entity_labels'].explode()`: Expands lists of tags in each row of the `entity_labels` column into individual entries.
   - `.unique()`: Identifies the distinct values from the expanded data.
   - `set(...)`: Ensures a collection of unique tags is created.

2. **Mapping Tags to IDs**:
   - `tag2id`: Assigns each tag a unique numerical ID using `enumerate()`.

3. **Reverse Mapping**:
   - `id2tag`: Allows conversion back from numerical IDs to their respective tags for interpretability.


In [ ]:
unique_tags

{'College Name',
 'Companies worked at',
 'Degree',
 'Designation',
 'Email Address',
 'Empty',
 'Graduation Year',
 'Location',
 'Name',
 'Skills',
 'UNKNOWN',
 'Years of Experience'}

<a id='3-2-8'></a>

### 3.2.8. Creating and Padding the Tag Array

To prepare the dataset for training, we will create an array of tags corresponding to the cleaned dataset. The tag array will be used to match the named-entities to numerical IDs defined in `tag2id`.

#### Key Challenges
- **Sequence Lengths**: Input sequences can vary in length and may exceed the maximum allowed by the network.
  - If the sequence exceeds the desired maximum length, it needs to be truncated.
  - If the sequence is shorter, padding must be applied to ensure uniform sequence length.
  
- **Solution**:
  We will use the [Keras padding API](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/sequence/pad_sequences) to handle this. The API facilitates truncating longer sequences and padding shorter ones with a specified value.

#### Implementation

In [ ]:
# Define maximum length for sequences
MAX_LEN = 512

# Extract the labels from the DataFrame
labels = cleanedDF['entity_labels'].values.tolist()

# Map each tag to its ID and pad/truncate sequences to the specified MAX_LEN
tags = pad_sequences(
    [[tag2id.get(l) for l in lab] for lab in labels],  # Convert tags to numerical IDs
    maxlen=MAX_LEN,                                    # Maximum sequence length
    value=tag2id["Empty"],                             # Value used for padding
    padding="post",                                    # Add padding at the end of sequences
    dtype="long",                                      # Data type for numerical IDs
    truncating="post"                                  # Truncate longer sequences from the end
)

#### Explanation

1. **Mapping Tags to IDs**:
   - Each label (tag) in `labels` is mapped to its corresponding ID using `tag2id`.
   - This ensures that the model processes numerical representations of the tags.

2. **Padding and Truncation**:
   - **Padding**: If a sequence has fewer tags than `MAX_LEN`, zeros (mapped from `"Empty"`) are appended to make it uniform.
   - **Truncating**: If a sequence has more tags than `MAX_LEN`, it is cut off at the end.

3. **Result**:
   - The `tags` array contains sequences of length exactly `MAX_LEN`, where each tag is represented by its numerical ID.

In [ ]:
tags

array([[10, 10,  6, ...,  1,  1,  1],
       [10, 10,  1, ...,  1,  1,  1],
       [10, 10, 10, ...,  1,  2,  1],
       ...,
       [10, 10,  6, ...,  1,  1,  1],
       [10, 10,  6, ...,  1,  1,  1],
       [10, 10,  6, ...,  1,  1,  1]])

<a id='3-2-9'></a>

### 3.2.9. Tokenize and Align inputs with Library

Before feeding textual data to a Transformer-based model, we need to tokenize the inputs using a compatible tokenizer. Huggingface has many [Transformer tokenizer](https://huggingface.co/transformers/main_classes/tokenizer.html). For this implementation, we use the [🤗 DistilBERT fast tokenizer](https://huggingface.co/transformers/model_doc/distilbert.html). The tokenizer ensures consistency with the model by:
- Splitting words into subwords when necessary.
- Standardizing sequence lengths to 512 tokens, padding shorter sequences with zeros.
- Truncating sequences exceeding the maximum length.

#### GPU Configuration
We configure TensorFlow to limit GPU memory usage to prevent resource exhaustion.

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_virtual_device_configuration(
            gpu, [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=4096)]
        )

#### Load the Tokenizer
The tokenizer must match the pre-trained Transformer model being used.

In [ ]:
# Load the DistilBERT fast tokenizer
# Tries local Kaggle path first, falls back to HuggingFace Hub
import os
TOKENIZER_PATH = '/kaggle/input/tokenizer'
if not os.path.exists(TOKENIZER_PATH):
    TOKENIZER_PATH = 'distilbert-base-uncased'  # fallback to HuggingFace Hub
    print('⚠️  Local tokenizer not found. Loading from HuggingFace Hub...')
tokenizer = DistilBertTokenizerFast.from_pretrained(TOKENIZER_PATH)
print(f'✅ Tokenizer loaded from: {TOKENIZER_PATH}')

⚠️  Local tokenizer not found. Loading from HuggingFace Hub...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

✅ Tokenizer loaded from: distilbert-base-uncased


#### Addressing Subword Tokenization and Label Alignment

Transformer tokenizers, like DistilBERT's, often split words into subwords. For example, a word like "Africa" might become `['A', '##frica']`. This creates misalignment between the original tags and the tokenized inputs.

##### Key Challenges:
1. **Special Tokens**: Tokenizers introduce special tokens (e.g., `[CLS]`, `[SEP]`) that have no corresponding labels.
 - `[CLS]` Token acts as a placeholder for the entire input sequence. In classification tasks, the final hidden state corresponding to `[CLS]` is used as the sequence representation for prediction.
 - `[SEP]` Token separates segments within a sequence or marks the end of the sequence. In single-sequence tasks, `[SEP]` is placed at the end of the sequence.
 ```
Input Text: "Transformers are amazing!"
Tokenized Sequence: ['[CLS]', 'Transformers', 'are', 'amazing', '!', '[SEP]']
```
2. **Subword Splitting**: Words split into multiple tokens must share a single label.
3. **Loss Function Exclusion**: Tokens irrelevant to training, such as special tokens or additional subwords, should be ignored during loss computation.

##### Alignment Strategy
We implement a `tokenize_and_align_labels()` function that:
- Tokenizes the inputs with `truncation=True` to enforce the maximum sequence length.
- Aligns the tokenized output to the original tags using the tokenizer's `word_ids()` method.
- Assigns:
  - $-100$ to special tokens, preventing them from contributing to the loss.
  - The label of the first subtoken to all subtokens of a word, depending on the `label_all_tokens` flag.

In [ ]:
label_all_tokens = True  # Controls how subtokens are labeled

def tokenize_and_align_labels(tokenizer, examples, tags):
    # Tokenize the inputs with truncation and padding
    tokenized_inputs = tokenizer(
        examples,
        truncation=True,
        is_split_into_words=False, # whether the input text (`examples`) is already split into words (True) or is a single string (False) that needs tokenization.
        padding='max_length', # Pads all sequences to the `max_length` specified
        max_length=512
    )

    labels = []
    for i, label in enumerate(tags):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Map subtokens to original words ids
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Handle special tokens
            if word_idx is None:
                label_ids.append(-100)
            # Assign label to the first token of a word
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # Assign label to subsequent subtokens or ignore
            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    # Add the aligned labels to the tokenized inputs
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

#### Tokenization and Alignment:

In [ ]:
# Tokenize and align labels for the dataset
tokenized_data = tokenize_and_align_labels(tokenizer, df_data['content'].values.tolist(), tags)

#### Output:
- `tokenized_data` contains tokenized inputs and aligned labels:
  - `input_ids`: Tokenized sequence IDs.
  - `labels`: Aligned labels matching the tokenized sequence.

#### Creating Train and Test Datasets

With aligned data, we can create TensorFlow datasets for training and evaluation.

In [ ]:
# Create a TensorFlow dataset
train_dataset = tf.data.Dataset.from_tensor_slices((
    tokenized_data['input_ids'],
    tokenized_data['labels']
))
test = tokenized_data

<a id="4"></a>
<h1 style="background: #FFC07F; border: 0; color: #2F2E41;
    box-shadow: 4px 4px 8px rgba(0, 0, 0, 0.3);
    padding: 10px; border-radius: 10px; margin: 15px 0;">
    <center style="color: #2F2E41;">4. Optimization</center>
</h1>

In [ ]:
model = TFDistilBertForTokenClassification.from_pretrained("distilbert-base-uncased", num_labels=len(unique_tags))

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForTokenClassification: ['vocab_transform.weight', 'vocab_layer_norm.weight', 'vocab_projector.bias', 'vocab_layer_norm.bias', 'vocab_transform.bias']
- This IS expected if you are initializing TFDistilBertForTokenClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForTokenClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForTokenClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able t

In [ ]:
# Print the model configuration
print("Model Configuration:")
print(model.config)

Model Configuration:
DistilBertConfig {
  "_name_or_path": "distilbert-base-uncased",
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "dim": 768,
  "dropout": 0.1,
  "hidden_dim": 3072,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3",
    "4": "LABEL_4",
    "5": "LABEL_5",
    "6": "LABEL_6",
    "7": "LABEL_7",
    "8": "LABEL_8",
    "9": "LABEL_9",
    "10": "LABEL_10",
    "11": "LABEL_11"
  },
  "initializer_range": 0.02,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_10": 10,
    "LABEL_11": 11,
    "LABEL_2": 2,
    "LABEL_3": 3,
    "LABEL_4": 4,
    "LABEL_5": 5,
    "LABEL_6": 6,
    "LABEL_7": 7,
    "LABEL_8": 8,
    "LABEL_9": 9
  },
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": 

In [ ]:
from transformers import AdamWeightDecay

# Define the optimizer with a custom learning rate
optimizer = AdamWeightDecay(learning_rate=1e-5, weight_decay_rate=0.01)

In [ ]:
model.compile(optimizer=optimizer, loss=model.hf_compute_loss, metrics=['accuracy']) # can also use any keras loss fn
model.fit(train_dataset.batch(4),
          epochs=150,
          batch_size=4)

Epoch 1/150
12/55 [=====>........................] - ETA: 10:35 - loss: 1.6979 - accuracy: 0.6638

<a id="4b"></a>
<h1 style="background: #4C72B0; border: 0; color: white; box-shadow: 4px 4px 8px rgba(0,0,0,0.3); padding: 10px; border-radius: 10px; margin: 15px 0;">
    <center>4B. BERT NER with Early Stopping (NEW)</center>
</h1>

In [ ]:
# ============================================================
# SECTION 4B: BERT NER Model with Early Stopping (NEW)
# ============================================================

# ---- Config ----
BERT_MODEL_NAME = "bert-base-uncased"
BERT_MAX_LEN    = 512
BERT_EPOCHS     = 30          # max epochs - early stopping will cut it short
BERT_BATCH_SIZE = 4
BERT_LR         = 2e-5
EARLY_STOP_PATIENCE = 3       # stop if val_loss doesn't improve for 3 epochs

print("Loading BERT tokenizer...")
bert_tokenizer = BertTokenizerFast.from_pretrained(BERT_MODEL_NAME)

# Re-use the same tokenize_and_align_labels function (already defined above)
print("Tokenizing data for BERT...")
tokenized_bert = tokenize_and_align_labels(bert_tokenizer,
                                            df_data['content'].values.tolist(),
                                            tags)

# Train/Val split (80/20)
n = len(tokenized_bert['input_ids'])
split = int(0.8 * n)

bert_train_ds = tf.data.Dataset.from_tensor_slices((
    tokenized_bert['input_ids'][:split],
    tokenized_bert['labels'][:split]
)).batch(BERT_BATCH_SIZE)

bert_val_ds = tf.data.Dataset.from_tensor_slices((
    tokenized_bert['input_ids'][split:],
    tokenized_bert['labels'][split:]
)).batch(BERT_BATCH_SIZE)

# ---- Build BERT model ----
print("Loading BERT for token classification...")
bert_model = TFBertForTokenClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=len(unique_tags)
)

from transformers import AdamWeightDecay
bert_optimizer = AdamWeightDecay(learning_rate=BERT_LR, weight_decay_rate=0.01)
bert_model.compile(optimizer=bert_optimizer, loss=bert_model.hf_compute_loss, metrics=['accuracy'])

# ---- Early Stopping Callback ----
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=EARLY_STOP_PATIENCE,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    verbose=1,
    min_lr=1e-7
)

print(f"Training BERT NER (max {BERT_EPOCHS} epochs, early stop patience={EARLY_STOP_PATIENCE})...")
bert_history = bert_model.fit(
    bert_train_ds,
    validation_data=bert_val_ds,
    epochs=BERT_EPOCHS,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print("✅ BERT training complete!")


<a id="4c"></a>
<h1 style="background: #DD8452; border: 0; color: white; box-shadow: 4px 4px 8px rgba(0,0,0,0.3); padding: 10px; border-radius: 10px; margin: 15px 0;">
    <center>4C. LoRA Fine-tuning (NEW)</center>
</h1>

In [ ]:
# ============================================================
# SECTION 4C: LoRA Fine-tuning for NER (NEW)
# ============================================================
# LoRA (Low-Rank Adaptation) adds small trainable rank-decomposition
# matrices to the attention layers - keeps 99%+ of weights FROZEN,
# trains only ~0.5-1% of parameters. Faster, less GPU memory.

import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn

# ---- Config ----
LORA_MODEL_NAME  = "bert-base-uncased"
LORA_EPOCHS      = 20
LORA_BATCH_SIZE  = 8
LORA_LR          = 3e-4     # LoRA can use higher LR than full fine-tune
LORA_R           = 16       # rank of the decomposition matrices
LORA_ALPHA       = 32       # scaling factor
LORA_DROPOUT     = 0.1
LORA_PATIENCE    = 3        # early stopping patience

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ---- Load base model & apply LoRA ----
print("Loading base model for LoRA...")
lora_base_model = AutoModelForTokenClassification.from_pretrained(
    LORA_MODEL_NAME,
    num_labels=len(unique_tags),
    id2label=id2tag,
    label2id=tag2id
)

lora_config = LoraConfig(
    task_type=TaskType.TOKEN_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["query", "value"],  # Apply LoRA to attention query & value
    bias="none"
)

lora_model = get_peft_model(lora_base_model, lora_config)
lora_model.print_trainable_parameters()  # Shows how few params are trained
lora_model = lora_model.to(device)

# ---- Tokenize for PyTorch ----
lora_tokenizer = AutoTokenizer.from_pretrained(LORA_MODEL_NAME)
tokenized_lora = tokenize_and_align_labels(lora_tokenizer,
                                           df_data['content'].values.tolist(),
                                           tags)

input_ids_t = torch.tensor(tokenized_lora['input_ids'], dtype=torch.long)
labels_t    = torch.tensor(tokenized_lora['labels'],    dtype=torch.long)

split = int(0.8 * len(input_ids_t))
train_ds_lora = TensorDataset(input_ids_t[:split], labels_t[:split])
val_ds_lora   = TensorDataset(input_ids_t[split:], labels_t[split:])

train_loader = DataLoader(train_ds_lora, batch_size=LORA_BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds_lora,   batch_size=LORA_BATCH_SIZE)

# ---- Optimizer ----
optimizer_lora = torch.optim.AdamW(lora_model.parameters(), lr=LORA_LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_lora, patience=2, factor=0.5)

# ---- Training loop with Early Stopping ----
best_val_loss = float('inf')
patience_counter = 0
lora_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print(f"Training LoRA model (max {LORA_EPOCHS} epochs)...")

for epoch in range(LORA_EPOCHS):
    # --- Train ---
    lora_model.train()
    total_loss, total_correct, total_tokens = 0, 0, 0

    for batch_ids, batch_labels in train_loader:
        batch_ids    = batch_ids.to(device)
        batch_labels = batch_labels.to(device)

        optimizer_lora.zero_grad()
        outputs = lora_model(input_ids=batch_ids, labels=batch_labels)
        loss = outputs.loss
        loss.backward()
        nn.utils.clip_grad_norm_(lora_model.parameters(), 1.0)
        optimizer_lora.step()

        total_loss += loss.item()
        preds = outputs.logits.argmax(-1)
        mask  = batch_labels != -100
        total_correct += (preds[mask] == batch_labels[mask]).sum().item()
        total_tokens  += mask.sum().item()

    train_loss = total_loss / len(train_loader)
    train_acc  = total_correct / total_tokens if total_tokens > 0 else 0

    # --- Validate ---
    lora_model.eval()
    val_loss_total, val_correct, val_tokens = 0, 0, 0
    with torch.no_grad():
        for batch_ids, batch_labels in val_loader:
            batch_ids    = batch_ids.to(device)
            batch_labels = batch_labels.to(device)
            outputs = lora_model(input_ids=batch_ids, labels=batch_labels)
            val_loss_total += outputs.loss.item()
            preds = outputs.logits.argmax(-1)
            mask  = batch_labels != -100
            val_correct += (preds[mask] == batch_labels[mask]).sum().item()
            val_tokens  += mask.sum().item()

    val_loss = val_loss_total / len(val_loader)
    val_acc  = val_correct / val_tokens if val_tokens > 0 else 0

    lora_history['train_loss'].append(train_loss)
    lora_history['val_loss'].append(val_loss)
    lora_history['train_acc'].append(train_acc)
    lora_history['val_acc'].append(val_acc)

    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}/{LORA_EPOCHS} | "
          f"train_loss: {train_loss:.4f} train_acc: {train_acc:.4f} | "
          f"val_loss: {val_loss:.4f} val_acc: {val_acc:.4f}")

    # --- Early Stopping ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save best LoRA weights
        lora_model.save_pretrained("/tmp/best_lora_ner")
        lora_tokenizer.save_pretrained("/tmp/best_lora_ner")
        print(f"  💾 Best model saved (val_loss={val_loss:.4f})")
    else:
        patience_counter += 1
        print(f"  ⚠️  No improvement ({patience_counter}/{LORA_PATIENCE})")
        if patience_counter >= LORA_PATIENCE:
            print(f"  🛑 Early stopping triggered at epoch {epoch+1}")
            break

print("✅ LoRA training complete!")


<a id="4d"></a>
<h1 style="background: #55A868; border: 0; color: white; box-shadow: 4px 4px 8px rgba(0,0,0,0.3); padding: 10px; border-radius: 10px; margin: 15px 0;">
    <center>4D. BERT vs LoRA Comparison (NEW)</center>
</h1>

In [ ]:
# ============================================================
# SECTION 4D: BERT vs LoRA — Comparison (NEW)
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("BERT Full Fine-tune vs LoRA: Training Comparison", fontsize=16, fontweight='bold')

# --- BERT Loss ---
ax = axes[0, 0]
ax.plot(bert_history.history['loss'],     label='BERT train loss', color='#4C72B0', linewidth=2)
ax.plot(bert_history.history['val_loss'], label='BERT val loss',   color='#4C72B0', linewidth=2, linestyle='--')
ax.set_title("BERT — Loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(True, alpha=0.3)

# --- BERT Accuracy ---
ax = axes[0, 1]
ax.plot(bert_history.history['accuracy'],     label='BERT train acc', color='#4C72B0', linewidth=2)
ax.plot(bert_history.history['val_accuracy'], label='BERT val acc',   color='#4C72B0', linewidth=2, linestyle='--')
ax.set_title("BERT — Accuracy")
ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
ax.legend(); ax.grid(True, alpha=0.3)

# --- LoRA Loss ---
ax = axes[1, 0]
ax.plot(lora_history['train_loss'], label='LoRA train loss', color='#DD8452', linewidth=2)
ax.plot(lora_history['val_loss'],   label='LoRA val loss',   color='#DD8452', linewidth=2, linestyle='--')
ax.set_title("LoRA — Loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(True, alpha=0.3)

# --- LoRA Accuracy ---
ax = axes[1, 1]
ax.plot(lora_history['train_acc'], label='LoRA train acc', color='#DD8452', linewidth=2)
ax.plot(lora_history['val_acc'],   label='LoRA val acc',   color='#DD8452', linewidth=2, linestyle='--')
ax.set_title("LoRA — Accuracy")
ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("bert_vs_lora_training.png", dpi=150, bbox_inches='tight')
plt.show()

# ---- Summary Table ----
bert_final_val_acc  = bert_history.history.get('val_accuracy', [0])[-1]
bert_final_val_loss = bert_history.history.get('val_loss',     [0])[-1]
lora_final_val_acc  = lora_history['val_acc'][-1]
lora_final_val_loss = lora_history['val_loss'][-1]
bert_epochs_run     = len(bert_history.history['loss'])
lora_epochs_run     = len(lora_history['train_loss'])

summary_df = pd.DataFrame({
    'Model':          ['BERT Full Fine-tune', 'BERT + LoRA'],
    'Trainable Params': ['~110M', f'~{LORA_R * 2 * 768 * 12 // 1_000_000:.1f}M (LoRA only)'],
    'Epochs Run':     [bert_epochs_run, lora_epochs_run],
    'Final Val Loss': [f'{bert_final_val_loss:.4f}', f'{lora_final_val_loss:.4f}'],
    'Final Val Acc':  [f'{bert_final_val_acc:.4f}',  f'{lora_final_val_acc:.4f}'],
    'Early Stopped':  ['Yes' if bert_epochs_run < BERT_EPOCHS else 'No',
                       'Yes' if lora_epochs_run < LORA_EPOCHS else 'No']
})

print("\n📊 BERT vs LoRA Summary:")
print(summary_df.to_string(index=False))


<a id="5"></a>
<h1 style="background: #FFC07F; border: 0; color: #2F2E41;
    box-shadow: 4px 4px 8px rgba(0, 0, 0, 0.3);
    padding: 10px; border-radius: 10px; margin: 15px 0;">
    <center style="color: #2F2E41;">5. Predictions</center>
</h1>

<a id="5-1"></a>

## 5.1. Tokenization and Prediction Using Transformer Models

In this section, we explore the tokenization of textual input and the application of a Transformer-based model to generate token-level predictions. This process forms the foundation for tasks such as Named Entity Recognition (NER). Below, we outline the methodology, including tokenization, model application, and prediction decoding.

<a id="5-1-1"></a>

### 5.1.1. Tokenization

Tokenization is a critical preprocessing step where textual input is converted into a format suitable for model processing. Transformer models, such as DistilBERT, require input sequences to be tokenized and padded or truncated to a consistent length.



In [ ]:
text = "Ananya Chavan\nlecturer - oracle tutorials\n\nMumbai, Maharashtra - Email me on Indeed: indeed.com/r/Ananya-\nChavan/738779ab71971a96\n\nSeeking a responsible job with an opportunity for professional challenges and utilize my skills\nup to its extreme.\n\nWORK EXPERIENCE\n\nlecturer\n\nOracle tutorials -  Mumbai, Maharashtra -\n\nApril 2016 to Present\n\nfor computer science (STD 11th and 12th) (2 years)\n➢ Worked at \"Dr.Babasaheb Ambedkar College, Chembur (W) \" as a lecturer for • B.Sc. (Computer\nScience & Information Technology)\n• F.Y.J.C. (Computer Science & I.T.)\n• S.Y.J.C. (Computer Science & I.T.)\n➢ Worked at \"LIVE\" as a Head of the IT Department and Lecturer for Web designing.\n➢ Worked at \"Kohinoor College Of Hotel Management\" as visiting lecturer for SEM I.\n➢ Working at \"ORACLE TUTORIALS\" as a lecturer for computer science (STD 11th and 12th)\n\nEDUCATION\n\nMCA\n\nMumbai University -  Mumbai, Maharashtra\n\nB.Sc. in Com.Sci\n\nMumbai University -  Mumbai, Maharashtra\n\nSKILLS\n\nSEARCH ENGINE MARKETING (2 years), SEM (2 years), ACCESS (Less than 1 year), AJAX (Less\nthan 1 year), APACHE (Less than 1 year)\n\nADDITIONAL INFORMATION\n\nTechnical skills:\nLanguages: C, C++, Java (J2EE),\nWeb Component APIS:: Jdbc, Servlet, JSP.\nFrameworks: Spring 4 & Struts 2\nORM Framework: Hibernate\nWeb Development: Html5, CSS3, Java Script, Ajax &JQuery, Angular Js\n\nhttps://www.indeed.com/r/Ananya-Chavan/738779ab71971a96?isid=rex-download&ikw=download-top&co=IN\nhttps://www.indeed.com/r/Ananya-Chavan/738779ab71971a96?isid=rex-download&ikw=download-top&co=IN\n\n\nApplication Servers: Apache Tomcat,\nIDE: Eclipse, Netbeans\nDatabase: Ms-Access, Mysql\nOperating Systems: Windows 7, 8, 10\nFTP Client: Filezilla\nVersioning Tools: Git\n\nProject Details:\n\n\"Real Estate Application\" (Client: Global Realtor PVT. LTD Pune)\nFront-End: Java (J2EE), JDBC, Servlet, JSP, Jquery.\nBack end: Mysql.\nDuration: 6 Month (Internship)\nCompany Name: AryanTech India Pvt. Ltd. Pune\nMy Role: Developer as Trainee.\nModule: Module 4.\nDescription: Developed as a MCA Final SEM Project for\n\"Global Realtors PVT.LTD, Hinjewadi, Pune.\"\nThe Real Estate Web Application is an interactive, effective and revenue-generating website\ndesigned for the Real Estate Industry. The main objective of this application is to help the Real\nEstate Company to display unlimited number of property listings on the website.\n\n\"Beauty Parlor Management System\" (B.Sc. (Com.Sci.))\nTool: VB 6.0\nLanguage: VB\nDatabase: MS-Access\nOperating System: Windows XP\nThe Beauty Parlor Management System is an easy and effective system to use. The main features\nof this system are to avoid manual work and keep storing all appointments of customers.\n\n\"Web Designing Project (Reptiles.com) \" (B.Sc. (Com.Sci.))\nLanguage: HTML and ASP\nTool: Dreamweaver 8.0\nDatabase: MS-Access\nOperating System: Windows XP\nThe Reptiles.com is a simple informative site. The main features of this system are to give all\ninformation of Snakes."

In [ ]:
# Replace newline characters with a space
processed_text = text.replace("\n", " ")

# Optionally, you can also strip extra spaces if needed
processed_text = " ".join(processed_text.split())

# Print the processed text
print(processed_text)

In [ ]:
text
inputs = tokenizer(
    processed_text,
    return_tensors="tf",        # Outputs tensors compatible with TensorFlow
    truncation=True,            # Truncate input sequences exceeding max_length
    is_split_into_words=False,  # Process the text as a single string
    padding="max_length",       # Pad sequences to the maximum length
    max_length=512              # Set the maximum sequence length
)
input_ids = inputs["input_ids"]

<a id="5-1-2"></a>

### 5.1.2. Model Prediction

Once tokenized, the input is passed to the Transformer model to generate logits, which represent the model's raw predictions for each token.

In [ ]:
output = model(inputs).logits
prediction = np.argmax(output, axis=2)
print(prediction)

- **Logits:** The raw outputs of the model before applying any activation functions. Each token in the input sequence receives a vector of scores corresponding to possible classes.
- **Prediction:** The class index with the highest score is selected for each token using `np.argmax`.

<a id="5-1-3"></a>

### 5.1.3. Decoding the Predictions

The predicted indices can be mapped to human-readable labels using a predefined mapping, allowing interpretation of the results.

In [ ]:
decoded_labels = [[id2tag[idx] for idx in seq] for seq in prediction]
print(decoded_labels)

In [ ]:
# Retrieve tokens from the tokenizer to display alongside the predictions
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

# Truncate tokens and labels to match the actual input length
actual_length = len(tokens)  # Actual length based on the input text (no padding)
tokens = tokens[:actual_length]
decoded_labels = decoded_labels[0][:actual_length]

# Prepare the result for display
result = list(zip(tokens, decoded_labels))

# Display the text and predicted labels in a readable format
for token, label in result:
    print(f"Token: {token} | Predicted Label: {label}")

<a id="5-2"></a>

## 5.2. Model Evaluation and Performance Analysis

This section outlines the steps involved in aligning tokenized inputs and true labels, predicting outputs using a trained Transformer model, and evaluating the model's performance using metrics and visualization.

<a id="5-2-1"></a>

## 5.2.1. Alignment of True Labels

The true labels corresponding to the dataset are aligned with the tokenized input to ensure consistency during evaluation.

In [ ]:
true_labels = [
    [id2tag.get(true_index, "Empty") for true_index in test['labels'][i]]
    for i in range(len(test['labels']))
]
np.array(true_labels).shape

<a id="5-2-2"></a>

## 5.2.2. Model Predictions

The model is applied to the tokenized dataset, and predictions are generated. These predictions are then reshaped and processed to align with the true labels.

In [ ]:
output = model.predict(train_dataset)

predictions = np.argmax(output['logits'].reshape(220, -1, 12), axis=-1)

predictions.shape

<a id="5-2-3"></a>

### 5.2.3. Visualizing Label Distributions

The distribution of true and predicted labels is visualized using histograms to assess label imbalances and prediction consistency.

#### Code for True Label Distribution

In [ ]:
from collections import Counter

# Flatten the true labels
true_labels_flat = np.array(true_labels).flatten()

# Count occurrences of each label
label_counts = Counter(true_labels_flat)

# Prepare data for Seaborn
labels = list(label_counts.keys())
counts = list(label_counts.values())

# Set Seaborn style for a professional look
sns.set_theme(style="whitegrid")

# Create a bar plot
plt.figure(figsize=(12, 6))
ax = sns.barplot(x=labels, y=counts, palette="viridis")

# Customize plot aesthetics
ax.set_title("Distribution of True Labels", fontsize=16, weight="bold")
ax.set_xlabel("Entity Labels", fontsize=14)
ax.set_ylabel("Count", fontsize=14)
plt.xticks(rotation=45, fontsize=12)  # Rotate x-axis labels for better readability
plt.yticks(fontsize=12)
plt.tight_layout()

# Show the plot
plt.show()

#### Code for Predicted Label Distribution

In [ ]:
pred_labels = [
    [id2tag.get(index, "Empty") for index in predictions[i]]
    for i in range(len(predictions))
]

In [ ]:
# Flatten the predicted labels
pred_labels_flat = np.array(pred_labels).flatten()

# Count occurrences of each label
pred_label_counts = Counter(pred_labels_flat)

# Prepare data for Seaborn
pred_labels_keys = list(pred_label_counts.keys())
pred_labels_values = list(pred_label_counts.values())

# Set Seaborn style for a professional look
sns.set_theme(style="whitegrid")

# Create a bar plot
plt.figure(figsize=(12, 6))
ax = sns.barplot(x=pred_labels_keys, y=pred_labels_values, palette="rocket")

# Customize plot aesthetics
ax.set_title("Distribution of Predicted Labels", fontsize=16, weight="bold")
ax.set_xlabel("Entity Labels", fontsize=14)
ax.set_ylabel("Count", fontsize=14)
plt.xticks(rotation=45, fontsize=12)  # Rotate x-axis labels for better readability
plt.yticks(fontsize=12)
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(true_labels_flat, pred_labels_flat))

<a id="6pdf"></a>
<h1 style="background: #C44E52; border: 0; color: white; box-shadow: 4px 4px 8px rgba(0,0,0,0.3); padding: 10px; border-radius: 10px; margin: 15px 0;">
    <center>6. PDF Upload &amp; Test (NEW)</center>
</h1>

In [ ]:
# ============================================================
# SECTION 6: PDF UPLOAD & TEST (NEW)
# ============================================================
# Upload any PDF (resume, CV, document) and run NER on it.
# Works with single-page and multi-page PDFs.

try:
    import pdfplumber
    PDF_LIB = 'pdfplumber'
except ImportError:
    try:
        import PyPDF2
        PDF_LIB = 'PyPDF2'
    except ImportError:
        PDF_LIB = None
        print("⚠️ No PDF library found. Run the requirements cell first.")


def extract_text_from_pdf(pdf_path):
    """Extract text from PDF using pdfplumber (preferred) or PyPDF2."""
    text_pages = []

    if PDF_LIB == 'pdfplumber':
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                t = page.extract_text()
                if t:
                    text_pages.append(t)
    elif PDF_LIB == 'PyPDF2':
        import PyPDF2
        with open(pdf_path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                t = page.extract_text()
                if t:
                    text_pages.append(t)
    else:
        raise ImportError("Install pdfplumber: pip install pdfplumber")

    full_text = " ".join(text_pages)
    full_text = re.sub(r'\s+', ' ', full_text).strip()
    return full_text, len(text_pages)


def run_ner_on_text(text, model_choice='distilbert'):
    """
    Run NER on any text string.
    model_choice: 'distilbert' (original), 'bert', or 'lora'
    """
    text = re.sub(r'\s+', ' ', text.replace('\n', ' ')).strip()

    if model_choice == 'distilbert':
        tok = tokenizer
        mdl = model
        framework = 'tf'
    elif model_choice == 'bert':
        tok = bert_tokenizer
        mdl = bert_model
        framework = 'tf'
    elif model_choice == 'lora':
        tok = lora_tokenizer
        mdl = lora_model
        framework = 'torch'
    else:
        raise ValueError("model_choice must be 'distilbert', 'bert', or 'lora'")

    inputs = tok(
        text,
        return_tensors="tf" if framework == 'tf' else "pt",
        truncation=True,
        is_split_into_words=False,
        padding='max_length',
        max_length=512
    )

    if framework == 'tf':
        logits = mdl(inputs).logits
        preds  = np.argmax(logits, axis=2)[0]
    else:
        import torch
        device_ = next(mdl.parameters()).device
        input_ids_ = inputs['input_ids'].to(device_)
        with torch.no_grad():
            logits = mdl(input_ids=input_ids_).logits
        preds = logits.argmax(-1)[0].cpu().numpy()

    tokens_out    = tok.convert_ids_to_tokens(inputs['input_ids'][0])
    decoded_preds = [id2tag.get(int(p), 'O') for p in preds]

    # Merge subword tokens and group consecutive same-label spans
    results = []
    current_word, current_label = '', None

    for token, label in zip(tokens_out, decoded_preds):
        if token in ('[CLS]', '[SEP]', '[PAD]', '<s>', '</s>', '<pad>'):
            if current_word and current_label and current_label != 'Empty':
                results.append({'text': current_word.strip(), 'label': current_label})
            current_word, current_label = '', None
            continue

        if token.startswith('##'):
            current_word += token[2:]
        else:
            if current_word and current_label and current_label != 'Empty':
                results.append({'text': current_word.strip(), 'label': current_label})
            current_word  = token
            current_label = label

    # Group by label
    from itertools import groupby
    entity_map = {}
    for r in results:
        entity_map.setdefault(r['label'], []).append(r['text'])

    return entity_map, results


def test_pdf(pdf_path, model_choice='distilbert'):
    """Full pipeline: PDF → text extraction → NER → pretty print."""
    print(f"\n📄 Processing: {pdf_path}")
    print(f"🤖 Model: {model_choice}")
    print("=" * 60)

    text, num_pages = extract_text_from_pdf(pdf_path)
    print(f"Pages extracted: {num_pages}")
    print(f"Text length: {len(text)} chars")
    print(f"Preview: {text[:200]}...\n")

    entity_map, raw_results = run_ner_on_text(text, model_choice=model_choice)

    print("🏷️  Extracted Entities:")
    print("-" * 40)
    for label, entities in sorted(entity_map.items()):
        unique_ents = list(dict.fromkeys(entities))[:10]  # deduplicate, show max 10
        print(f"  {label:20s}: {', '.join(unique_ents)}")

    return entity_map, raw_results


# ============================================================
# 🔼 UPLOAD YOUR PDF HERE
# ============================================================
# Option A: Google Colab file upload
try:
    from google.colab import files
    print("📎 Please upload a PDF file:")
    uploaded = files.upload()
    pdf_filename = list(uploaded.keys())[0]
    print(f"✅ Uploaded: {pdf_filename}")

    # Test with all 3 models
    for model_name in ['distilbert', 'bert', 'lora']:
        try:
            entities, _ = test_pdf(pdf_filename, model_choice=model_name)
        except Exception as e:
            print(f"⚠️ {model_name} failed: {e}")

except Exception:
    # Option B: Local path — change this to your PDF path
    PDF_PATH = "your_resume.pdf"
    if os.path.exists(PDF_PATH):
        entities, _ = test_pdf(PDF_PATH, model_choice='distilbert')
    else:
        print("ℹ️  To test: run test_pdf('your_file.pdf', model_choice='distilbert')")
        print("   Or on Colab, run the upload cell above.")


In [ ]:
# ============================================================
# SECTION 6B: Compare all 3 models on same PDF (NEW)
# ============================================================

def compare_models_on_pdf(pdf_path):
    """Run all 3 models on the same PDF and display a comparison table."""
    text, _ = extract_text_from_pdf(pdf_path)

    results_all = {}
    model_names = ['distilbert', 'bert', 'lora']

    for mn in model_names:
        try:
            entity_map, _ = run_ner_on_text(text, model_choice=mn)
            results_all[mn] = entity_map
        except Exception as e:
            results_all[mn] = {'ERROR': [str(e)]}

    # Build comparison dataframe
    all_labels = sorted(set(
        label for em in results_all.values() for label in em.keys()
    ))

    rows = []
    for label in all_labels:
        row = {'Entity Type': label}
        for mn in model_names:
            ents = results_all.get(mn, {}).get(label, [])
            row[mn] = ', '.join(list(dict.fromkeys(ents))[:5])
        rows.append(row)

    cmp_df = pd.DataFrame(rows)
    print("\n📊 Model Comparison on PDF:")
    print(cmp_df.to_string(index=False))
    return cmp_df

# Uncomment after uploading PDF:
# compare_models_on_pdf(pdf_filename)


<a id="6"></a>
<h1 style="background: #FFC07F; border: 0; color: #2F2E41;
    box-shadow: 4px 4px 8px rgba(0, 0, 0, 0.3);
    padding: 10px; border-radius: 10px; margin: 15px 0;">
    <center style="color: #2F2E41;">6. Conclusion</center>
</h1>

### **Key Takeaways**

- **Named-Entity Recognition (NER):**  
  NER is a powerful tool for detecting and classifying named entities within text. It is widely used in applications such as resume parsing, sentiment analysis of customer reviews, and processing browsing histories.

- **Preprocessing Requirements:**  
  Before inputting textual data into a Transformer model, it is essential to preprocess the data using the tokenizer associated with the pretrained model. This ensures compatibility and optimizes model performance.